# Interseção SICOR Cerrado × Terras Indígenas por ID_GLEBA2

Este notebook cruza o arquivo `sicor_cerrado_2021_2025_enriquecido_detalhado_com_id_gleba2.csv` com o shapefile `tis_poligonais.shp`.

## Regra principal

- cada `ID_GLEBA2` distinto é submetido à interseção espacial somente uma vez;
- quando o mesmo `ID_GLEBA2` aparece em várias linhas, o resultado calculado é reaproveitado;
- antes do processamento, o notebook valida se um mesmo `ID_GLEBA2` está sempre associado à mesma geometria;
- a deduplicação técnica usa a chave `ID_GLEBA2 + TI_ID_INTERNO`, e não `REF_BACEN`.

## Saídas

1. **Resultado espacial único:** uma linha por `ID_GLEBA2 × Terra Indígena`.
2. **Resultado detalhado expandido:** associa a resposta espacial a todas as linhas originais que possuem aquele `ID_GLEBA2`.
3. **Resumo por ID_GLEBA2:** contém todos os IDs, inclusive os que não intersectaram Terras Indígenas.

Os atributos do shapefile são preservados com o prefixo `TI_`. Assim, uma geometria repetida não é testada novamente, mas nenhum vínculo com `REF_BACEN`, ordem, índice ou ano é perdido.


In [1]:
# ============================================================
# 1. INSTALAR, IMPORTAR E MONTAR O GOOGLE DRIVE
# ============================================================

%pip install -q --upgrade geopandas pyogrio shapely

from google.colab import drive
drive.mount('/content/drive')

from pathlib import Path
import gc
import re
import unicodedata
import warnings

import numpy as np
import pandas as pd
import geopandas as gpd

from shapely import force_2d, intersection as shapely_intersection, make_valid, set_precision
from shapely.errors import GEOSException

warnings.filterwarnings('ignore', category=UserWarning)

pd.set_option('display.max_columns', None)
pd.set_option('display.width', None)
pd.set_option('display.max_colwidth', 180)

print('Ambiente preparado.')


Mounted at /content/drive
Ambiente preparado.


In [2]:
# ============================================================
# 2. CAMINHOS E PARÂMETROS
# ============================================================

ARQUIVO_SICOR = Path(
    "/content/drive/MyDrive/Colab Notebooks/fluxo/Envio/"
    "Arquivos gerados e relatorios/Base04_enriquecimento_detalhado/"
    "sicor_cerrado_2021_2025_enriquecido_detalhado_com_id_gleba2.csv"
)

ARQUIVO_TI = Path(
    "/content/drive/MyDrive/Colab Notebooks/fluxo/Envio/"
    "Base/tis_poligonais.shp"
)

# Pasta nova para não sobrescrever resultados anteriores.
PASTA_SAIDA = Path(
    "/content/drive/MyDrive/Colab Notebooks/fluxo/Envio/"
    "Arquivos gerados e relatorios/intersecoes/terras indígenas 2/"
    "resultados_por_id_gleba2"
)

PASTA_SAIDA.mkdir(parents=True, exist_ok=True)

ARQUIVO_INTERSECAO_POR_ID = (
    PASTA_SAIDA / "intersecao_tis_unica_por_id_gleba2.csv"
)

ARQUIVO_INTERSECAO_EXPANDIDA = (
    PASTA_SAIDA / "intersecao_sicor_tis_expandida_por_id_gleba2.csv"
)

ARQUIVO_RESUMO_POR_ID = (
    PASTA_SAIDA / "resumo_tis_por_id_gleba2.csv"
)

CRS_SICOR = "EPSG:4326"
CRS_AREA = "EPSG:5880"
TAMANHO_BLOCO = 20_000
SEPARADOR_SAIDA = ";"

for caminho in [ARQUIVO_SICOR, ARQUIVO_TI]:
    print(f"{'OK' if caminho.exists() else 'NÃO ENCONTRADO'} | {caminho}")

if not ARQUIVO_SICOR.exists():
    raise FileNotFoundError(f"CSV do SICOR não encontrado: {ARQUIVO_SICOR}")

if not ARQUIVO_TI.exists():
    raise FileNotFoundError(
        f"Shapefile de Terras Indígenas não encontrado: "
        f"{ARQUIVO_TI}"
    )

for extensao in [".dbf", ".shx", ".prj"]:
    componente = ARQUIVO_TI.with_suffix(extensao)
    print(f"{'OK' if componente.exists() else 'ATENÇÃO'} | {componente.name}")

print("\nPasta de saída:")
print(PASTA_SAIDA)


OK | /content/drive/MyDrive/Colab Notebooks/fluxo/Envio/Arquivos gerados e relatorios/Base04_enriquecimento_detalhado/sicor_cerrado_2021_2025_enriquecido_detalhado_com_id_gleba2.csv
OK | /content/drive/MyDrive/Colab Notebooks/fluxo/Envio/Base/tis_poligonais.shp
OK | tis_poligonais.dbf
OK | tis_poligonais.shx
OK | tis_poligonais.prj

Pasta de saída:
/content/drive/MyDrive/Colab Notebooks/fluxo/Envio/Arquivos gerados e relatorios/intersecoes/terras indígenas 2/resultados_por_id_gleba2


In [3]:
# ============================================================
# 3. FUNÇÕES AUXILIARES
# ============================================================

CODIFICACOES = ['utf-8-sig', 'utf-8', 'latin-1', 'cp1252']
SEPARADORES = [';', ',', '\t', '|']


def remover_acentos(texto):
    return ''.join(
        caractere
        for caractere in unicodedata.normalize('NFKD', str(texto))
        if not unicodedata.combining(caractere)
    )


def normalizar_nome_coluna(nome):
    nome = remover_acentos(nome).upper().strip().replace('#', '')
    nome = re.sub(r'[^A-Z0-9]+', '_', nome)
    return re.sub(r'_+', '_', nome).strip('_')


def normalizar_colunas(df):
    df = df.copy()
    df.columns = [normalizar_nome_coluna(coluna) for coluna in df.columns]

    if df.columns.duplicated().any():
        duplicadas = df.columns[df.columns.duplicated()].tolist()
        raise ValueError(f'A normalização produziu colunas duplicadas: {duplicadas}')

    return df


def detectar_csv(caminho):
    ultimo_erro = None

    for encoding in CODIFICACOES:
        for separador in SEPARADORES:
            try:
                teste = pd.read_csv(
                    caminho,
                    sep=separador,
                    encoding=encoding,
                    nrows=5,
                    dtype=str,
                    low_memory=False,
                )

                if teste.shape[1] <= 1:
                    continue

                teste = normalizar_colunas(teste)
                return encoding, separador, list(teste.columns)

            except Exception as erro:
                ultimo_erro = erro

    raise RuntimeError(
        'Não foi possível detectar a estrutura do CSV. '
        f'Último erro: {ultimo_erro}'
    )


def localizar_coluna(colunas, candidatos, obrigatoria=True):
    for candidato in [normalizar_nome_coluna(c) for c in candidatos]:
        if candidato in colunas:
            return candidato

    if obrigatoria:
        raise KeyError(
            'Nenhuma das colunas esperadas foi encontrada: '
            f'{[normalizar_nome_coluna(c) for c in candidatos]}'
        )

    return None


def padronizar_codigo(serie):
    return (
        serie.astype('string')
        .str.strip()
        .str.lstrip('\ufeff')
        .str.replace(r'\.0$', '', regex=True)
        .replace({'': pd.NA, 'NAN': pd.NA, 'NONE': pd.NA, '<NA>': pd.NA})
    )


def corrigir_geometrias(gdf):
    quantidade_entrada = len(gdf)

    gdf = gdf[gdf.geometry.notna() & ~gdf.geometry.is_empty].copy()
    mascara_invalida = ~gdf.geometry.is_valid
    quantidade_invalidas = int(mascara_invalida.sum())

    if quantidade_invalidas:
        try:
            gdf.loc[mascara_invalida, 'geometry'] = (
                gdf.loc[mascara_invalida, 'geometry'].make_valid()
            )
        except Exception:
            gdf.loc[mascara_invalida, 'geometry'] = (
                gdf.loc[mascara_invalida, 'geometry'].buffer(0)
            )

    gdf = gdf[
        gdf.geometry.notna()
        & ~gdf.geometry.is_empty
        & gdf.geometry.is_valid
    ].copy()

    quantidade_descartadas = quantidade_entrada - len(gdf)
    return gdf, quantidade_invalidas, quantidade_descartadas


def serie_wkt_para_geometria(serie):
    return gpd.GeoSeries.from_wkt(
        serie.astype('string'),
        crs=CRS_SICOR,
        on_invalid='ignore',
    )


def preparar_geometria_topologica(geometria, precisao=None):
    if geometria is None:
        return None

    try:
        if geometria.is_empty:
            return None
        geometria = force_2d(geometria)

        if not geometria.is_valid:
            geometria = make_valid(geometria)

        if precisao is not None:
            geometria = set_precision(
                geometria,
                grid_size=precisao,
                mode='valid_output',
            )

        if geometria is None or geometria.is_empty:
            return None

        if not geometria.is_valid:
            geometria = make_valid(geometria)

        return geometria

    except Exception:
        try:
            geometria = geometria.buffer(0)
            return None if geometria.is_empty else geometria
        except Exception:
            return None


def calcular_intersecoes_seguras(geometrias_esquerda, geometrias_direita, crs):
    resultados = []
    quantidade_recuperadas = 0
    quantidade_falhas = 0

    for geometria_esquerda, geometria_direita in zip(
        geometrias_esquerda.to_numpy(),
        geometrias_direita.to_numpy(),
    ):
        esquerda = preparar_geometria_topologica(geometria_esquerda, 0.01)
        direita = preparar_geometria_topologica(geometria_direita, 0.01)

        if esquerda is None or direita is None:
            resultados.append(None)
            quantidade_falhas += 1
            continue

        intersecao = None
        recuperada = False

        try:
            intersecao = shapely_intersection(esquerda, direita, grid_size=0.01)
        except (GEOSException, ValueError, TypeError):
            recuperada = True

            try:
                esquerda_10 = preparar_geometria_topologica(esquerda, 0.10)
                direita_10 = preparar_geometria_topologica(direita, 0.10)

                if esquerda_10 is not None and direita_10 is not None:
                    intersecao = shapely_intersection(
                        esquerda_10,
                        direita_10,
                        grid_size=0.10,
                    )
            except (GEOSException, ValueError, TypeError):
                intersecao = None

            if intersecao is None:
                try:
                    intersecao = esquerda.buffer(0).intersection(direita.buffer(0))
                except Exception:
                    intersecao = None

        if intersecao is None:
            resultados.append(None)
            quantidade_falhas += 1
            continue

        try:
            if not intersecao.is_empty and not intersecao.is_valid:
                intersecao = make_valid(intersecao)
        except Exception:
            try:
                intersecao = intersecao.buffer(0)
            except Exception:
                intersecao = None

        if intersecao is None:
            resultados.append(None)
            quantidade_falhas += 1
        else:
            resultados.append(intersecao)
            if recuperada:
                quantidade_recuperadas += 1

    return (
        gpd.GeoSeries(resultados, index=pd.RangeIndex(len(resultados)), crs=crs),
        quantidade_recuperadas,
        quantidade_falhas,
    )


In [4]:
# ============================================================
# 4. LER E PREPARAR AS TERRAS INDÍGENAS
# ============================================================

tis = gpd.read_file(ARQUIVO_TI, engine='pyogrio')
crs_original_ti = tis.crs
coluna_geometria_original = tis.geometry.name

if crs_original_ti is None:
    raise ValueError(
        'O shapefile tis_poligonais.shp não possui CRS. Verifique se o arquivo .prj '
        'está na mesma pasta do shapefile.'
    )

tis = normalizar_colunas(tis)
coluna_geometria_normalizada = normalizar_nome_coluna(coluna_geometria_original)

colunas_geometricas = [
    coluna
    for coluna in tis.columns
    if isinstance(tis[coluna].dtype, gpd.array.GeometryDtype)
]

if coluna_geometria_normalizada not in tis.columns:
    if not colunas_geometricas:
        raise KeyError('Nenhuma coluna geométrica foi localizada no shapefile de Terras Indígenas.')
    coluna_geometria_normalizada = colunas_geometricas[0]

tis = gpd.GeoDataFrame(
    tis,
    geometry=coluna_geometria_normalizada,
    crs=crs_original_ti,
)

if tis.geometry.name != 'geometry':
    tis = tis.rename_geometry('geometry')

quantidade_ti_original = len(tis)
tis, invalidas_ti, descartadas_ti = corrigir_geometrias(tis)
tis = tis.reset_index(drop=True)
tis['TI_ID_INTERNO'] = np.arange(1, len(tis) + 1)

renomear_ti = {
    coluna: f'TI_{coluna}'
    for coluna in tis.columns
    if coluna not in {'geometry', 'TI_ID_INTERNO'}
}
tis = tis.rename(columns=renomear_ti).set_geometry('geometry')
tis = tis.to_crs(CRS_SICOR).reset_index(drop=True)

tis_area = tis.to_crs(CRS_AREA)
tis['TI_AREA_TOTAL_HA'] = tis_area.geometry.area / 10_000

COLUNAS_TI_SAIDA = [coluna for coluna in tis.columns if coluna != 'geometry']

print('Feições originais de Terras Indígenas:', f'{quantidade_ti_original:,}')
print('Geometrias inválidas corrigidas:', f'{invalidas_ti:,}')
print('Geometrias descartadas:', f'{descartadas_ti:,}')
print('Feições válidas finais:', f'{len(tis):,}')
print('CRS espacial:', tis.crs)
print('CRS para áreas:', CRS_AREA)

display(tis.head())


Feições originais de Terras Indígenas: 661
Geometrias inválidas corrigidas: 0
Geometrias descartadas: 0
Feições válidas finais: 661
CRS espacial: EPSG:4326
CRS para áreas: EPSG:5880


,TI_GID,TI_TERRAI_COD,TI_TERRAI_NOM,TI_ETNIA_NOME,TI_MUNICIPIO,TI_UF_SIGLA,TI_SUPERFICIE,TI_FASE_TI,TI_MODALIDADE,TI_REESTUDO_T,TI_CR,TI_FAIXA_FRON,TI_UNDADM_COD,TI_UNDADM_NOM,TI_UNDADM_SIG,TI_DOMINIO_UN,TI_DATA_ATUAL,TI_EPSG,geometry,TI_ID_INTERNO,TI_AREA_TOTAL_HA
0,1,101,Acapuri de Cima,Kokama,Fonte Boa,AM,18393.9411,Regularizada,Tradicionalmente ocupada,None,COORDENACAO REGIONAL DO ALTO SOLIMOES,Não,3.020200e+10,COORDENACAO REGIONAL DO ALTO SOLIMOES,CR-AS,t,05/09/2023,4674,"POLYGON ((-66.88359 -2.53752, -66.87457 -2.53756, -66.86559 -2.53759, -66.85658 -2.53765, -66.84762 -2.53767, -66.83858 -2.53777, -66.82967 -2.53775, -66.82062 -2.53785, -66.81...",1,18606.501826
1,2,201,Acimã,Apurinã,Lábrea,AM,40686.0340,Regularizada,Tradicionalmente ocupada,None,COORDENACAO REGIONAL MÉDIO PURUS,Não,3.020200e+10,COORDENACAO REGIONAL MÉDIO PURUS,CR-Mpur,t,None,4674,"POLYGON ((-66.30929 -7.79995, -66.30929 -7.79994, -66.3083 -7.79866, -66.30751 -7.79817, -66.30667 -7.7981, -66.30574 -7.79832, -66.30421 -7.79881, -66.30356 -7.79903, -66.3029...",2,41528.704020
2,3,401,Água Preta/Inari,Apurinã,Pauini,AM,139763.6705,Regularizada,Tradicionalmente ocupada,None,COORDENACAO REGIONAL MÉDIO PURUS,Não,3.020200e+10,COORDENACAO REGIONAL MÉDIO PURUS,CR-Mpur,t,None,4674,"POLYGON ((-67.29764 -7.33958, -67.29713 -7.33895, -67.29659 -7.3383, -67.2953 -7.3364, -67.29504 -7.33601, -67.29355 -7.33317, -67.29344 -7.33295, -67.29051 -7.33009, -67.28914...",3,143388.290407
3,4,501,Águas Belas,Pataxó,Prado,BA,1189.0649,Regularizada,Tradicionalmente ocupada,None,COORDENACAO REGIONAL SUL DA BAHIA,Não,3.020200e+10,COORDENACAO REGIONAL SUL DA BAHIA,CR-SBA,t,14/04/2026,4674,"POLYGON ((-39.30549 -16.90609, -39.305 -16.90619, -39.30447 -16.90611, -39.30384 -16.906, -39.30341 -16.90583, -39.30302 -16.90576, -39.30271 -16.90567, -39.3024 -16.9056, -39....",4,1242.656438
4,5,601,Aconã,Tingui-Botó,Traipu,AL,267.7862,Regularizada,Reserva Indígena,None,COORDENACAO REGIONAL NORDESTE I,Não,3.020200e+10,COORDENACAO REGIONAL NORDESTE I,CR-NE-I,t,30/04/2026,4674,"POLYGON ((-36.94513 -10.07113, -36.94382 -10.06967, -36.94378 -10.06961, -36.94223 -10.06762, -36.93794 -10.06205, -36.9369 -10.06282, -36.93572 -10.06037, -36.93326 -10.05816,...",5,278.299678


In [5]:
# ============================================================
# 5. DETECTAR E VALIDAR AS COLUNAS DO SICOR
# ============================================================

encoding_sicor, separador_sicor, colunas_sicor = detectar_csv(ARQUIVO_SICOR)

COLUNA_ID_GLEBA2 = localizar_coluna(
    colunas_sicor,
    ['ID_GLEBA2', 'ID_GLEBA_2'],
)

COLUNA_GEOMETRIA_SICOR = localizar_coluna(
    colunas_sicor,
    ['GT_GEOMETRIA', 'GT_GEOMETRIA_REPRESENTATIVA', 'GEOMETRIA', 'GEOMETRY', 'WKT'],
)

COLUNA_REF_BACEN = localizar_coluna(
    colunas_sicor,
    ['REF_BACEN', '#REF_BACEN', 'CD_REF_BACEN'],
    obrigatoria=False,
)

print('Codificação:', encoding_sicor)
print('Separador:', repr(separador_sicor))
print('Coluna-chave:', COLUNA_ID_GLEBA2)
print('Coluna de geometria:', COLUNA_GEOMETRIA_SICOR)
print('Coluna REF_BACEN:', COLUNA_REF_BACEN)
print('Quantidade de colunas:', len(colunas_sicor))

if COLUNA_ID_GLEBA2 == COLUNA_GEOMETRIA_SICOR:
    raise RuntimeError('A coluna do ID e a coluna geométrica não podem ser a mesma.')


Codificação: utf-8-sig
Separador: ';'
Coluna-chave: ID_GLEBA2
Coluna de geometria: GT_GEOMETRIA
Coluna REF_BACEN: REF_BACEN
Quantidade de colunas: 27


In [6]:
# ============================================================
# 6. INTERSECTAR UMA ÚNICA VEZ CADA ID_GLEBA2
# ============================================================

# Guarda somente uma assinatura numérica da geometria de cada ID.
# Isso permite reconhecer IDs já processados sem manter todos os WKT na memória.
assinatura_geometria_por_id = {}
resultados_intersecao = []

total_linhas_lidas = 0
total_linhas_com_id_e_geometria = 0
total_ids_novos = 0
total_repeticoes_puladas = 0
total_ids_geometria_valida = 0
total_geometrias_invalidas_corrigidas = 0
total_geometrias_descartadas = 0
total_candidatos_espaciais = 0
total_intersecoes_validas = 0
total_intersecoes_recuperadas = 0
total_falhas_topologicas = 0
total_contatos_sem_area = 0

leitor = pd.read_csv(
    ARQUIVO_SICOR,
    sep=separador_sicor,
    encoding=encoding_sicor,
    dtype=str,
    chunksize=TAMANHO_BLOCO,
    low_memory=False,
)

for numero_bloco, bloco in enumerate(leitor, start=1):
    bloco = normalizar_colunas(bloco)
    total_linhas_lidas += len(bloco)

    bloco[COLUNA_ID_GLEBA2] = padronizar_codigo(bloco[COLUNA_ID_GLEBA2])
    bloco[COLUNA_GEOMETRIA_SICOR] = bloco[COLUNA_GEOMETRIA_SICOR].astype('string').str.strip()

    bloco = bloco.dropna(
        subset=[COLUNA_ID_GLEBA2, COLUNA_GEOMETRIA_SICOR]
    ).copy()

    total_linhas_com_id_e_geometria += len(bloco)

    if bloco.empty:
        print(f'Bloco {numero_bloco:03d}: sem ID_GLEBA2 e geometria válidos')
        continue

    bloco['_HASH_GEOMETRIA'] = (
        pd.util.hash_pandas_object(
            bloco[COLUNA_GEOMETRIA_SICOR],
            index=False,
        )
        .astype('uint64')
    )

    # Validação dentro do próprio bloco.
    geometrias_por_id_no_bloco = bloco.groupby(
        COLUNA_ID_GLEBA2,
        dropna=False,
    )['_HASH_GEOMETRIA'].nunique()

    ids_inconsistentes = geometrias_por_id_no_bloco[
        geometrias_por_id_no_bloco > 1
    ].index.tolist()

    if ids_inconsistentes:
        raise ValueError(
            'Um mesmo ID_GLEBA2 foi associado a geometrias diferentes. '
            f'Primeiros IDs inconsistentes: {ids_inconsistentes[:10]}'
        )

    representantes = bloco.drop_duplicates(
        subset=[COLUNA_ID_GLEBA2],
        keep='first',
    ).reset_index(drop=True)

    mascara_id_novo = []

    # Validação também contra IDs que apareceram em blocos anteriores.
    for id_atual, hash_atual in representantes[
        [COLUNA_ID_GLEBA2, '_HASH_GEOMETRIA']
    ].itertuples(index=False, name=None):
        hash_atual = int(hash_atual)

        if id_atual in assinatura_geometria_por_id:
            if assinatura_geometria_por_id[id_atual] != hash_atual:
                raise ValueError(
                    f'O ID_GLEBA2 {id_atual} apareceu com geometrias diferentes '
                    'em blocos distintos do CSV.'
                )
            mascara_id_novo.append(False)
        else:
            assinatura_geometria_por_id[id_atual] = hash_atual
            mascara_id_novo.append(True)

    representantes_novos = representantes.loc[
        mascara_id_novo
    ].copy().reset_index(drop=True)

    total_ids_novos += len(representantes_novos)
    total_repeticoes_puladas += len(bloco) - len(representantes_novos)

    if representantes_novos.empty:
        print(
            f'Bloco {numero_bloco:03d}: lidas={len(bloco):,} | '
            'todos os IDs já haviam sido processados'
        )
        del bloco, representantes, representantes_novos
        gc.collect()
        continue

    geometria = serie_wkt_para_geometria(
        representantes_novos[COLUNA_GEOMETRIA_SICOR]
    )

    gdf_ids = gpd.GeoDataFrame(
        representantes_novos[
            [COLUNA_ID_GLEBA2, COLUNA_GEOMETRIA_SICOR]
        ].copy(),
        geometry=geometria,
        crs=CRS_SICOR,
    )

    gdf_ids, invalidas_corrigidas, descartadas = corrigir_geometrias(gdf_ids)
    gdf_ids = gdf_ids.reset_index(drop=True)

    total_geometrias_invalidas_corrigidas += invalidas_corrigidas
    total_geometrias_descartadas += descartadas
    total_ids_geometria_valida += len(gdf_ids)

    if gdf_ids.empty:
        print(
            f'Bloco {numero_bloco:03d}: IDs novos={len(representantes_novos):,} | '
            'nenhuma geometria válida'
        )
        del bloco, representantes, representantes_novos, gdf_ids
        gc.collect()
        continue

    candidatos = gpd.sjoin(
        gdf_ids,
        tis,
        how='inner',
        predicate='intersects',
    ).reset_index(drop=True)

    total_candidatos_espaciais += len(candidatos)

    if candidatos.empty:
        print(
            f'Bloco {numero_bloco:03d}: lidas={len(bloco):,} | '
            f'IDs novos={len(representantes_novos):,} | interseções=0 | '
            f'repetições puladas={len(bloco) - len(representantes_novos):,}'
        )
        del bloco, representantes, representantes_novos, gdf_ids, candidatos
        gc.collect()
        continue

    candidatos['GEOMETRIA_TI'] = (
        candidatos['index_right'].map(tis.geometry)
    )

    candidatos = candidatos.dropna(
        subset=['GEOMETRIA_TI']
    ).reset_index(drop=True)

    geometria_gleba_wgs84 = gpd.GeoSeries(
        candidatos.geometry.to_numpy(),
        index=pd.RangeIndex(len(candidatos)),
        crs=CRS_SICOR,
    )

    geometria_ti_wgs84 = gpd.GeoSeries(
        candidatos['GEOMETRIA_TI'].to_numpy(),
        index=pd.RangeIndex(len(candidatos)),
        crs=CRS_SICOR,
    )

    geometria_gleba_area = geometria_gleba_wgs84.to_crs(CRS_AREA)
    geometria_ti_area = geometria_ti_wgs84.to_crs(CRS_AREA)

    geometria_gleba_area = gpd.GeoSeries(
        [preparar_geometria_topologica(g, 0.01) for g in geometria_gleba_area],
        index=pd.RangeIndex(len(geometria_gleba_area)),
        crs=CRS_AREA,
    )

    geometria_ti_area = gpd.GeoSeries(
        [preparar_geometria_topologica(g, 0.01) for g in geometria_ti_area],
        index=pd.RangeIndex(len(geometria_ti_area)),
        crs=CRS_AREA,
    )

    (
        geometria_intersecao_area,
        recuperadas_bloco,
        falhas_bloco,
    ) = calcular_intersecoes_seguras(
        geometria_gleba_area,
        geometria_ti_area,
        CRS_AREA,
    )

    total_intersecoes_recuperadas += recuperadas_bloco
    total_falhas_topologicas += falhas_bloco

    candidatos['AREA_GLEBA_HA_CALCULADA'] = (
        geometria_gleba_area.area.to_numpy() / 10_000
    )
    candidatos['AREA_INTERSECAO_HA'] = (
        geometria_intersecao_area.area.to_numpy() / 10_000
    )
    candidatos['PERCENTUAL_GLEBA_INTERSECTADA'] = np.where(
        candidatos['AREA_GLEBA_HA_CALCULADA'] > 0,
        candidatos['AREA_INTERSECAO_HA']
        / candidatos['AREA_GLEBA_HA_CALCULADA']
        * 100,
        np.nan,
    )

    mascara_area_positiva = (
        candidatos['AREA_INTERSECAO_HA'].fillna(0).gt(0).to_numpy()
    )
    total_contatos_sem_area += int((~mascara_area_positiva).sum())

    if not mascara_area_positiva.any():
        print(
            f'Bloco {numero_bloco:03d}: candidatos={len(candidatos):,} | '
            'nenhuma interseção com área positiva'
        )
        del (
            bloco, representantes, representantes_novos, gdf_ids, candidatos,
            geometria_gleba_wgs84, geometria_ti_wgs84,
            geometria_gleba_area, geometria_ti_area,
            geometria_intersecao_area, mascara_area_positiva,
        )
        gc.collect()
        continue

    candidatos = candidatos.loc[mascara_area_positiva].reset_index(drop=True)

    geometria_intersecao_area_valida = gpd.GeoSeries(
        geometria_intersecao_area.to_numpy()[mascara_area_positiva],
        index=pd.RangeIndex(int(mascara_area_positiva.sum())),
        crs=CRS_AREA,
    )
    geometria_intersecao_wgs84 = geometria_intersecao_area_valida.to_crs(CRS_SICOR)

    candidatos['GEOMETRIA_TI_WKT'] = [
        g.wkt if g is not None and not g.is_empty else pd.NA
        for g in candidatos['GEOMETRIA_TI']
    ]
    candidatos['GEOMETRIA_INTERSECAO_WKT'] = [
        g.wkt if g is not None and not g.is_empty else pd.NA
        for g in geometria_intersecao_wgs84
    ]
    candidatos['TIPO_IMPEDIMENTO'] = 'Terra Indígena'

    colunas_resultado = [
        COLUNA_ID_GLEBA2,
        *COLUNAS_TI_SAIDA,
        'AREA_GLEBA_HA_CALCULADA',
        'AREA_INTERSECAO_HA',
        'PERCENTUAL_GLEBA_INTERSECTADA',
        'GEOMETRIA_TI_WKT',
        'GEOMETRIA_INTERSECAO_WKT',
        'TIPO_IMPEDIMENTO',
    ]

    resultado_bloco = pd.DataFrame(candidatos[colunas_resultado])
    resultados_intersecao.append(resultado_bloco)
    total_intersecoes_validas += len(resultado_bloco)

    print(
        f'Bloco {numero_bloco:03d}: lidas={len(bloco):,} | '
        f'IDs novos={len(representantes_novos):,} | '
        f'repetições puladas={len(bloco) - len(representantes_novos):,} | '
        f'candidatos={len(mascara_area_positiva):,} | '
        f'interseções com área={len(resultado_bloco):,}'
    )

    del (
        bloco, representantes, representantes_novos, gdf_ids, candidatos,
        geometria_gleba_wgs84, geometria_ti_wgs84,
        geometria_gleba_area, geometria_ti_area,
        geometria_intersecao_area, geometria_intersecao_area_valida,
        geometria_intersecao_wgs84, mascara_area_positiva, resultado_bloco,
    )
    gc.collect()

print('\nPROCESSAMENTO ESPACIAL CONCLUÍDO')
print('=' * 90)
print('Linhas lidas:', f'{total_linhas_lidas:,}')
print('Linhas com ID e geometria:', f'{total_linhas_com_id_e_geometria:,}')
print('ID_GLEBA2 distintos:', f'{len(assinatura_geometria_por_id):,}')
print('IDs submetidos ao teste:', f'{total_ids_novos:,}')
print('Repetições que não foram testadas novamente:', f'{total_repeticoes_puladas:,}')
print('IDs com geometria válida:', f'{total_ids_geometria_valida:,}')
print('Geometrias inválidas corrigidas:', f'{total_geometrias_invalidas_corrigidas:,}')
print('Geometrias descartadas:', f'{total_geometrias_descartadas:,}')
print('Candidatos espaciais:', f'{total_candidatos_espaciais:,}')
print('Interseções recuperadas:', f'{total_intersecoes_recuperadas:,}')
print('Falhas topológicas:', f'{total_falhas_topologicas:,}')
print('Contatos sem área positiva:', f'{total_contatos_sem_area:,}')
print('Interseções válidas ID_GLEBA2 × Terra Indígena:', f'{total_intersecoes_validas:,}')


Bloco 001: lidas=20,000 | IDs novos=18,663 | repetições puladas=1,337 | candidatos=49 | interseções com área=49
Bloco 002: lidas=20,000 | IDs novos=17,690 | repetições puladas=2,310 | candidatos=22 | interseções com área=22
Bloco 003: lidas=20,000 | IDs novos=17,256 | repetições puladas=2,744 | candidatos=20 | interseções com área=20
Bloco 004: lidas=20,000 | IDs novos=17,061 | repetições puladas=2,939 | candidatos=15 | interseções com área=15
Bloco 005: lidas=20,000 | IDs novos=16,706 | repetições puladas=3,294 | candidatos=24 | interseções com área=23
Bloco 006: lidas=20,000 | IDs novos=14,364 | repetições puladas=5,636 | candidatos=29 | interseções com área=29
Bloco 007: lidas=20,000 | IDs novos=15,135 | repetições puladas=4,865 | candidatos=14 | interseções com área=14
Bloco 008: lidas=20,000 | IDs novos=15,230 | repetições puladas=4,770 | candidatos=30 | interseções com área=30
Bloco 009: lidas=20,000 | IDs novos=14,954 | repetições puladas=5,046 | candidatos=25 | interseções com 

In [7]:
# ============================================================
# 7. CONSOLIDAR E SALVAR O RESULTADO ÚNICO POR ID_GLEBA2
# ============================================================

if resultados_intersecao:
    intersecoes_por_id = pd.concat(
        resultados_intersecao,
        ignore_index=True,
        sort=False,
    )
else:
    intersecoes_por_id = pd.DataFrame(
        columns=[COLUNA_ID_GLEBA2, 'TI_ID_INTERNO']
    )

del resultados_intersecao
gc.collect()

quantidade_antes = len(intersecoes_por_id)

if not intersecoes_por_id.empty:
    intersecoes_por_id = (
        intersecoes_por_id
        .sort_values('AREA_INTERSECAO_HA', ascending=False)
        .drop_duplicates(
            subset=[COLUNA_ID_GLEBA2, 'TI_ID_INTERNO'],
            keep='first',
        )
        .reset_index(drop=True)
    )

print(
    'Duplicações técnicas ID_GLEBA2 × Terra Indígena removidas:',
    f'{quantidade_antes - len(intersecoes_por_id):,}',
)

intersecoes_por_id.to_csv(
    ARQUIVO_INTERSECAO_POR_ID,
    index=False,
    sep=SEPARADOR_SAIDA,
    decimal=',',
    encoding='utf-8-sig',
)

# O resumo inclui todos os IDs, inclusive os sem interseção.
todos_ids = pd.DataFrame({
    COLUNA_ID_GLEBA2: list(assinatura_geometria_por_id.keys())
})

if intersecoes_por_id.empty:
    resumo_por_id = todos_ids.copy()
    resumo_por_id['TEM_INTERSECAO_TI'] = False
    resumo_por_id['QUANTIDADE_TERRAS_INDIGENAS_INTERSECTADAS'] = 0
    resumo_por_id['AREA_INTERSECAO_TOTAL_HA'] = 0.0
    resumo_por_id['PERCENTUAL_MAXIMO_GLEBA_INTERSECTADA'] = 0.0
else:
    resumo_positivo = (
        intersecoes_por_id.groupby(COLUNA_ID_GLEBA2, dropna=False)
        .agg(
            QUANTIDADE_TERRAS_INDIGENAS_INTERSECTADAS=(
                'TI_ID_INTERNO', 'nunique'
            ),
            AREA_INTERSECAO_TOTAL_HA=('AREA_INTERSECAO_HA', 'sum'),
            PERCENTUAL_MAXIMO_GLEBA_INTERSECTADA=(
                'PERCENTUAL_GLEBA_INTERSECTADA', 'max'
            ),
        )
        .reset_index()
    )

    resumo_por_id = todos_ids.merge(
        resumo_positivo,
        on=COLUNA_ID_GLEBA2,
        how='left',
        validate='one_to_one',
    )

    resumo_por_id['TEM_INTERSECAO_TI'] = (
        resumo_por_id['QUANTIDADE_TERRAS_INDIGENAS_INTERSECTADAS']
        .notna()
    )
    resumo_por_id['QUANTIDADE_TERRAS_INDIGENAS_INTERSECTADAS'] = (
        resumo_por_id['QUANTIDADE_TERRAS_INDIGENAS_INTERSECTADAS']
        .fillna(0)
        .astype('int64')
    )
    resumo_por_id['AREA_INTERSECAO_TOTAL_HA'] = (
        resumo_por_id['AREA_INTERSECAO_TOTAL_HA'].fillna(0.0)
    )
    resumo_por_id['PERCENTUAL_MAXIMO_GLEBA_INTERSECTADA'] = (
        resumo_por_id['PERCENTUAL_MAXIMO_GLEBA_INTERSECTADA'].fillna(0.0)
    )

    resumo_por_id = resumo_por_id[
        [
            COLUNA_ID_GLEBA2,
            'TEM_INTERSECAO_TI',
            'QUANTIDADE_TERRAS_INDIGENAS_INTERSECTADAS',
            'AREA_INTERSECAO_TOTAL_HA',
            'PERCENTUAL_MAXIMO_GLEBA_INTERSECTADA',
        ]
    ]

resumo_por_id.to_csv(
    ARQUIVO_RESUMO_POR_ID,
    index=False,
    sep=SEPARADOR_SAIDA,
    decimal=',',
    encoding='utf-8-sig',
)

print('\nResultado único salvo em:')
print(ARQUIVO_INTERSECAO_POR_ID)
print('\nResumo de todos os IDs salvo em:')
print(ARQUIVO_RESUMO_POR_ID)
print('\nIDs com interseção:', f"{resumo_por_id['TEM_INTERSECAO_TI'].sum():,}")
print('IDs sem interseção:', f"{(~resumo_por_id['TEM_INTERSECAO_TI']).sum():,}")


Duplicações técnicas ID_GLEBA2 × Terra Indígena removidas: 0

Resultado único salvo em:
/content/drive/MyDrive/Colab Notebooks/fluxo/Envio/Arquivos gerados e relatorios/intersecoes/terras indígenas 2/resultados_por_id_gleba2/intersecao_tis_unica_por_id_gleba2.csv

Resumo de todos os IDs salvo em:
/content/drive/MyDrive/Colab Notebooks/fluxo/Envio/Arquivos gerados e relatorios/intersecoes/terras indígenas 2/resultados_por_id_gleba2/resumo_tis_por_id_gleba2.csv

IDs com interseção: 554
IDs sem interseção: 395,021


In [8]:
# ============================================================
# 8. DEVOLVER A RESPOSTA A TODAS AS LINHAS DO SICOR
# ============================================================

# A interseção não é recalculada aqui. O merge apenas replica a resposta
# de cada ID_GLEBA2 para todos os REF_BACEN, ordens, índices e anos ligados a ele.

colunas_espaciais_para_merge = [
    coluna
    for coluna in intersecoes_por_id.columns
    if coluna != COLUNA_ID_GLEBA2
]

escrever_cabecalho = True
total_linhas_originais_reprocessadas = 0
total_linhas_expandidas = 0

leitor_expandir = pd.read_csv(
    ARQUIVO_SICOR,
    sep=separador_sicor,
    encoding=encoding_sicor,
    dtype=str,
    chunksize=TAMANHO_BLOCO,
    low_memory=False,
)

for numero_bloco, bloco in enumerate(leitor_expandir, start=1):
    bloco = normalizar_colunas(bloco)
    bloco[COLUNA_ID_GLEBA2] = padronizar_codigo(bloco[COLUNA_ID_GLEBA2])
    total_linhas_originais_reprocessadas += len(bloco)

    if intersecoes_por_id.empty:
        expandido = pd.DataFrame()
    else:
        expandido = bloco.merge(
            intersecoes_por_id,
            on=COLUNA_ID_GLEBA2,
            how='inner',
            validate='many_to_many',
            suffixes=('', '_INTERSECAO'),
        )

    if expandido.empty:
        print(f'Bloco {numero_bloco:03d}: nenhuma linha com interseção')
        del bloco, expandido
        gc.collect()
        continue

    expandido.to_csv(
        ARQUIVO_INTERSECAO_EXPANDIDA,
        mode='w' if escrever_cabecalho else 'a',
        header=escrever_cabecalho,
        index=False,
        sep=SEPARADOR_SAIDA,
        decimal=',',
        encoding='utf-8-sig' if escrever_cabecalho else 'utf-8',
    )

    total_linhas_expandidas += len(expandido)
    escrever_cabecalho = False

    print(
        f'Bloco {numero_bloco:03d}: originais={len(bloco):,} | '
        f'linhas de interseção expandidas={len(expandido):,}'
    )

    del bloco, expandido
    gc.collect()

if escrever_cabecalho:
    # Nenhuma interseção: ainda cria um CSV vazio com cabeçalho conhecido.
    pd.DataFrame(
        columns=[*colunas_sicor, *colunas_espaciais_para_merge]
    ).to_csv(
        ARQUIVO_INTERSECAO_EXPANDIDA,
        index=False,
        sep=SEPARADOR_SAIDA,
        encoding='utf-8-sig',
    )

if total_linhas_originais_reprocessadas != total_linhas_lidas:
    raise RuntimeError(
        'A quantidade de linhas relidas para a expansão difere da '
        'quantidade lida na etapa espacial.'
    )

print('\nRESULTADO DETALHADO EXPANDIDO')
print('=' * 90)
print('Linhas originais relidas:', f'{total_linhas_originais_reprocessadas:,}')
print('Linhas finais de interseção:', f'{total_linhas_expandidas:,}')
print('Arquivo salvo em:')
print(ARQUIVO_INTERSECAO_EXPANDIDA)


Bloco 001: originais=20,000 | linhas de interseção expandidas=55
Bloco 002: originais=20,000 | linhas de interseção expandidas=29
Bloco 003: originais=20,000 | linhas de interseção expandidas=25
Bloco 004: originais=20,000 | linhas de interseção expandidas=17
Bloco 005: originais=20,000 | linhas de interseção expandidas=28
Bloco 006: originais=20,000 | linhas de interseção expandidas=35
Bloco 007: originais=20,000 | linhas de interseção expandidas=20
Bloco 008: originais=20,000 | linhas de interseção expandidas=36
Bloco 009: originais=20,000 | linhas de interseção expandidas=34
Bloco 010: originais=20,000 | linhas de interseção expandidas=25
Bloco 011: originais=20,000 | linhas de interseção expandidas=20
Bloco 012: originais=20,000 | linhas de interseção expandidas=28
Bloco 013: originais=20,000 | linhas de interseção expandidas=26
Bloco 014: originais=20,000 | linhas de interseção expandidas=21
Bloco 015: originais=20,000 | linhas de interseção expandidas=69
Bloco 016: originais=20,0

In [9]:
# ============================================================
# 9. CONFERÊNCIA FINAL
# ============================================================

print('ARQUIVOS GERADOS')
print('=' * 100)

for caminho in [
    ARQUIVO_INTERSECAO_POR_ID,
    ARQUIVO_INTERSECAO_EXPANDIDA,
    ARQUIVO_RESUMO_POR_ID,
]:
    print(f"{'OK' if caminho.exists() else 'NÃO CRIADO'} | {caminho}")

print('\nAMOSTRA DO RESULTADO ÚNICO POR ID_GLEBA2')
display(intersecoes_por_id.head(5))

print('\nAMOSTRA DO RESUMO, INCLUSIVE IDs SEM INTERSEÇÃO')
display(resumo_por_id.head(5))

print('\nESTATÍSTICAS')
print('ID_GLEBA2 distintos analisados:', f'{len(resumo_por_id):,}')
print('ID_GLEBA2 com interseção:', f"{resumo_por_id['TEM_INTERSECAO_TI'].sum():,}")
print('Pares ID_GLEBA2 × Terra Indígena:', f'{len(intersecoes_por_id):,}')

if not intersecoes_por_id.empty:
    print(
        'Área total das interseções únicas (ha):',
        f"{intersecoes_por_id['AREA_INTERSECAO_HA'].sum():,.4f}",
    )


ARQUIVOS GERADOS
OK | /content/drive/MyDrive/Colab Notebooks/fluxo/Envio/Arquivos gerados e relatorios/intersecoes/terras indígenas 2/resultados_por_id_gleba2/intersecao_tis_unica_por_id_gleba2.csv
OK | /content/drive/MyDrive/Colab Notebooks/fluxo/Envio/Arquivos gerados e relatorios/intersecoes/terras indígenas 2/resultados_por_id_gleba2/intersecao_sicor_tis_expandida_por_id_gleba2.csv
OK | /content/drive/MyDrive/Colab Notebooks/fluxo/Envio/Arquivos gerados e relatorios/intersecoes/terras indígenas 2/resultados_por_id_gleba2/resumo_tis_por_id_gleba2.csv

AMOSTRA DO RESULTADO ÚNICO POR ID_GLEBA2


,ID_GLEBA2,TI_GID,TI_TERRAI_COD,TI_TERRAI_NOM,TI_ETNIA_NOME,TI_MUNICIPIO,TI_UF_SIGLA,TI_SUPERFICIE,TI_FASE_TI,TI_MODALIDADE,TI_REESTUDO_T,TI_CR,TI_FAIXA_FRON,TI_UNDADM_COD,TI_UNDADM_NOM,TI_UNDADM_SIG,TI_DOMINIO_UN,TI_DATA_ATUAL,TI_EPSG,TI_ID_INTERNO,TI_AREA_TOTAL_HA,AREA_GLEBA_HA_CALCULADA,AREA_INTERSECAO_HA,PERCENTUAL_GLEBA_INTERSECTADA,GEOMETRIA_TI_WKT,GEOMETRIA_INTERSECAO_WKT,TIPO_IMPEDIMENTO
0,174917,360,36602,Porquinhos dos Canela-Apãnjekra,Kanela,Barra do Corda e Fernando Falcão e Formosa da Serra Negra e Mirador,MA,301000.0,Delimitada,Tradicionalmente ocupada,Reestudo,COORDENACAO REGIONAL MARANHÃO,Não,3.020200e+10,COORDENACAO REGIONAL MARANHÃO,CR-MA,f,28/08/2018,4674,360,304469.649528,5004.128414,2779.285741,55.539857,"POLYGON ((-45.29130029 -5.96543311, -45.29141912 -5.96648107, -45.29138997 -5.96831352, -45.29110718 -5.96962342, -45.29090097 -5.97009406, -45.29024215 -5.97119243, -45.290016...","MULTIPOLYGON (((-45.450434986727814 -6.303869978668874, -45.45055997171961 -6.306265000689239, -45.450410342379975 -6.3134159762548725, -45.45052199754387 -6.316695960847419, -...",Terra Indígena
1,77692,360,36602,Porquinhos dos Canela-Apãnjekra,Kanela,Barra do Corda e Fernando Falcão e Formosa da Serra Negra e Mirador,MA,301000.0,Delimitada,Tradicionalmente ocupada,Reestudo,COORDENACAO REGIONAL MARANHÃO,Não,3.020200e+10,COORDENACAO REGIONAL MARANHÃO,CR-MA,f,28/08/2018,4674,360,304469.649528,2774.122077,2557.095507,92.176748,"POLYGON ((-45.29130029 -5.96543311, -45.29141912 -5.96648107, -45.29138997 -5.96831352, -45.29110718 -5.96962342, -45.29090097 -5.97009406, -45.29024215 -5.97119243, -45.290016...","POLYGON ((-45.466194991897545 -6.348533976981777, -45.46099597994631 -6.3157209904760565, -45.46452001496734 -6.289243002560746, -45.45163496934557 -6.2897379656567605, -45.452...",Terra Indígena
2,174917,203,20702,Kanela Memortumré,Kanela,"Fernando Falcão,Barra do Corda",MA,100221.0,Delimitada,Tradicionalmente ocupada,None,COORDENACAO REGIONAL MARANHÃO,Não,3.020200e+10,COORDENACAO REGIONAL MARANHÃO,CR-MA,f,10/02/2025,4674,203,101487.284759,5004.128414,2224.842312,44.460136,"MULTIPOLYGON (((-44.94329259 -6.03579864, -44.95857815 -6.19585637, -44.96024356 -6.17724059, -44.98441022 -6.12974227, -44.97857933 -6.11335027, -44.96413177 -6.09890648, -44....","POLYGON ((-45.38404602893205 -6.309369984599059, -45.39034438005924 -6.291246823344664, -45.37280856689717 -6.253998250887383, -45.37218296033424 -6.25502199233382, -45.3721879...",Terra Indígena
3,251498,360,36602,Porquinhos dos Canela-Apãnjekra,Kanela,Barra do Corda e Fernando Falcão e Formosa da Serra Negra e Mirador,MA,301000.0,Delimitada,Tradicionalmente ocupada,Reestudo,COORDENACAO REGIONAL MARANHÃO,Não,3.020200e+10,COORDENACAO REGIONAL MARANHÃO,CR-MA,f,28/08/2018,4674,360,304469.649528,4341.408006,2189.051562,50.422618,"POLYGON ((-45.29130029 -5.96543311, -45.29141912 -5.96648107, -45.29138997 -5.96831352, -45.29110718 -5.96962342, -45.29090097 -5.97009406, -45.29024215 -5.97119243, -45.290016...","MULTIPOLYGON (((-45.399470027964476 -6.31063315982959, -45.414806570414434 -6.343216884761977, -45.42556497809705 -6.345336958075639, -45.440908991540496 -6.344313993167714, -4...",Terra Indígena
4,315242,360,36602,Porquinhos dos Canela-Apãnjekra,Kanela,Barra do Corda e Fernando Falcão e Formosa da Serra Negra e Mirador,MA,301000.0,Delimitada,Tradicionalmente ocupada,Reestudo,COORDENACAO REGIONAL MARANHÃO,Não,3.020200e+10,COORDENACAO REGIONAL MARANHÃO,CR-MA,f,28/08/2018,4674,360,304469.649528,4341.421437,2180.279481,50.220406,"POLYGON ((-45.29130029 -5.96543311, -45.29141912 -5.96648107, -45.29138997 -5.96831352, -45.29110718 -5.96962342, -45.29090097 -5.97009406, -45.29024215 -5.97119243, -45.290016...","MULTIPOLYGON (((-45.3995121967635 -6.310722747891313, -45.414806570414434 -6.343216884761977, -45.42556497809705 -6.345336958075639, -45.43974600760647 -6.345872990404547, -45....",Terra Indígena



AMOSTRA DO RESUMO, INCLUSIVE IDs SEM INTERSEÇÃO


,ID_GLEBA2,TEM_INTERSECAO_TI,QUANTIDADE_TERRAS_INDIGENAS_INTERSECTADAS,AREA_INTERSECAO_TOTAL_HA,PERCENTUAL_MAXIMO_GLEBA_INTERSECTADA
0,1,False,0,0.0,0.0
1,2,False,0,0.0,0.0
2,3,False,0,0.0,0.0
3,4,False,0,0.0,0.0
4,5,False,0,0.0,0.0



ESTATÍSTICAS
ID_GLEBA2 distintos analisados: 395,575
ID_GLEBA2 com interseção: 554
Pares ID_GLEBA2 × Terra Indígena: 569
Área total das interseções únicas (ha): 51,554.9376
